In [8]:
import pandas as pd

# 将参加“每日听写”项目的学校映射到学校总表里

In [9]:
### 先对所有学校数据进行预处理
# 读取需要标记的总学校名录
Sumschool_file = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\Ecoles_primaires_2024_2025.xlsx'
df_Sumschool =pd.read_excel(Sumschool_file, sheet_name=0)

##城市/乡村学校0-1处理
df_Sumschool = df_Sumschool[df_Sumschool['Milieu_ecole'].isin(['Rural', 'Urbain'])]
df_Sumschool = df_Sumschool.dropna(subset=['Milieu_ecole'])
Milieu_mapping = {
    'Rural': 0,
    'Urbain': 1
}
df_Sumschool['Milieu_ecole'] = df_Sumschool['Milieu_ecole'].replace(Milieu_mapping)
try:
    df_Sumschool['Milieu_ecole'] = df_Sumschool['Milieu_ecole'].astype(int)
except ValueError:
    print("警告：'Statut' 字段转换整数失败，可能存在未处理的非数值/非映射值。")



##公立/私立学校0-1处理
df_Sumschool = df_Sumschool[df_Sumschool['Statut'].isin(['Public', 'Privé'])]
df_Sumschool = df_Sumschool.dropna(subset=['Statut'])
Statut_mapping = {
    'Public': 0,
    'Privé': 1
}
df_Sumschool['Statut'] = df_Sumschool['Statut'].replace(Statut_mapping)
try:
    df_Sumschool['Statut'] = df_Sumschool['Statut'].astype(int)
except ValueError:
    print("警告：'Statut' 字段转换整数失败，可能存在未处理的非数值/非映射值。")


# 定义用于匹配的字段
TARGET_COLS = ['DRENA', 'Nom_ecole', 'Statut', 'Milieu_ecole']
PROJECT_COLS = ['Lib_DREN', 'Ecole', 'Statut', 'Milieu_ecole']

df_project = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final_correction.csv')
df_project_clean = df_project[PROJECT_COLS].apply(lambda x: x.dropna().astype(str).str.strip())
all_project_schools = set(df_project_clean.itertuples(index=False, name=None))#没有出现重复元素


# 创建用于比对的四元组集合
df_Sumschool_clean = df_Sumschool[TARGET_COLS].apply(lambda x: x.dropna().astype(str).str.strip())
school_tuples = set(df_Sumschool_clean.itertuples(index=False, name=None))

# 添加is_in_project列 #这里df_Sumschool里面用四个字段做筛选后仍然有重复行，重复行只要被纳入项目则全部计1
df_Sumschool['is_in_project'] = df_Sumschool_clean.apply(
    lambda row: 1 if tuple(row) in all_project_schools else 0, axis=1
)


# 输出1: 带有标记列的新 Excel 文件
output_excel = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Ecoles_primaires_2024_2025_Tagged.xlsx'
# 计算不同DRENA/IEPP的参与“每日听写”项目学校密度
df_Sumschool['project_density_by_dren'] = df_Sumschool.groupby('codedren')['is_in_project'].transform('mean')
df_Sumschool['project_density_by_iepp'] = df_Sumschool.groupby('codeIEPP')['is_in_project'].transform('mean')
df_Sumschool.to_excel(output_excel, index=False)



# 记录all_project_schools中在上传文件中没有出现过的四元组
missing_tuples = all_project_schools - school_tuples

# 将缺失的四元组转换为DataFrame并输出至CSV文件
if missing_tuples:
    missing_df = pd.DataFrame(list(missing_tuples), columns=PROJECT_COLS)
    missing_csv_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\missing_project_schools.csv'
    missing_df.to_csv(missing_csv_path, index=False)
    print(f"已输出缺失的四元组至CSV文件: {missing_csv_path}")
    print(f"缺失的四元组数量: {len(missing_tuples)}")
else:
    print("所有项目学校都在上传文件中存在，无需输出缺失文件。")


# 输出统计信息
print(f"\n统计信息:")
print(f"总学校数量（有重复）: {len(df_Sumschool)}")
print(f"总学校数量（无重复）: {len(school_tuples)}")
print(f"参与项目的学校数量: {df_Sumschool['is_in_project'].sum()}")
print(f"项目学校集合中的学校数量: {len(all_project_schools)}")
print(f"学校参与项目率: {df_Sumschool['is_in_project'].sum()/len(df_Sumschool)*100:.2f}%")



C:\Users\ASUS\AppData\Local\Temp\ipykernel_36860\2756018963.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_Sumschool['Milieu_ecole'] = df_Sumschool['Milieu_ecole'].replace(Milieu_mapping)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_36860\2756018963.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_Sumschool['Statut'] = df_Sumschool['Statut'].replace(Statut_mapping)


已输出缺失的四元组至CSV文件: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\missing_project_schools.csv
缺失的四元组数量: 117

统计信息:
总学校数量（有重复）: 19495
总学校数量（无重复）: 19269
参与项目的学校数量: 3027
项目学校集合中的学校数量: 3078
学校参与项目率: 15.53%


# HeckMan二阶段选择模型的第一阶段Probit模型

In [10]:
import numpy as np
import statsmodels.api as sm
from scipy.stats import norm
from pyfixest.estimation import feols
from sklearn.preprocessing import StandardScaler

# Load Selection Data (Stage 1: All Schools)
df_select = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Ecoles_primaires_2024_2025_Tagged.xlsx', sheet_name=0)

# Load Outcome Data (Stage 2: In project Schools)
df_outcome = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final_correction.csv')


print(f"Selection Data Loaded. N={len(df_select)}")
print(f"Outcome Data Loaded. N={len(df_outcome)}")

# --- Probit Selection Model & IMR Calculation ---

# Define Stage 1 variables
stage1_cols = ['is_in_project', 'Milieu_ecole', 'Statut', 
               'ecole_electrifie_Y_N', 'salle_multimedia_y_n', 
               'Connexion_internet_Y_N', 'project_density_by_iepp']

#在正式论文中更名：
# is_in_project->IfProject
# Milieu_ecole->SchLocation
# Statut->SchType
# ecole_electrifie_Y_N->ElectrifiedSch
# salle_multimedia_y_n->MultimediaRoom
# Connexion_internet_Y_N->InternetSch
# project_density_by_iepp->ProjectDensity


# Prepare X and Y for Probit
Y_select = df_select['is_in_project']
X_select = df_select[['Milieu_ecole', 'Statut', 'ecole_electrifie_Y_N', 
                      'salle_multimedia_y_n', 'Connexion_internet_Y_N', 
                      'project_density_by_iepp']]

# Add constant for the intercept
X_select = sm.add_constant(X_select)

# Fit Probit Model
print("\n--- Running Stage 1: Probit Selection Model ---")
try:
    probit_model = sm.Probit(Y_select, X_select).fit(disp=0)
    print(probit_model.summary())
    
    # Calculate Inverse Mills Ratio (IMR)
    # Get the linear prediction (z-score)
    z_scores = probit_model.predict(X_select, linear=True)
    
    # Calculate PDF and CDF
    pdf = norm.pdf(z_scores)
    cdf = norm.cdf(z_scores)
    
    # IMR formula: lambda = phi(z) / Phi(z)
    df_select['IMR'] = pdf / cdf
    df_select.to_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Ecoles_primaires_2024_2025_Tagged_n_IMR.xlsx', index=False)
    print("IMR calculated successfully.")
    
except Exception as e:
    print(f"Error in Stage 1: {e}")
    exit()

Selection Data Loaded. N=19495
Outcome Data Loaded. N=43632

--- Running Stage 1: Probit Selection Model ---
                          Probit Regression Results                           
Dep. Variable:          is_in_project   No. Observations:                19495
Model:                         Probit   Df Residuals:                    19488
Method:                           MLE   Df Model:                            6
Date:                Thu, 12 Mar 2026   Pseudo R-squ.:                  0.3767
Time:                        11:30:05   Log-Likelihood:                -5246.6
converged:                       True   LL-Null:                       -8416.8
Covariance Type:            nonrobust   LLR p-value:                     0.000
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -2.0270      0.027    -75.663      0.000      -2

g:\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)


IMR calculated successfully.


# 将IMR映射到参与“每日听写”的学校表中去

In [11]:
imr_file = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Ecoles_primaires_2024_2025_Tagged_n_IMR.xlsx'
df_imr = pd.read_excel(imr_file, sheet_name=0)

outcome_file = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final_correction.csv'
df_outcome = pd.read_csv(outcome_file)

print(f"数据加载完成。IMR源数据行数: {len(df_imr)}, 结果数据行数: {len(df_outcome)}")


# 定义清洗函数：转字符串、去除首尾空格
def clean_key(series):
    return series.astype(str).str.strip()

# 处理 IMR 源数据 (df_imr)
# 对应字段: DRENA, Nom_ecole, Statut, Milieu_ecole
df_imr['key_dren'] = clean_key(df_imr['DRENA'])
df_imr['key_ecole'] = clean_key(df_imr['Nom_ecole'])
df_imr['Statut'] = pd.to_numeric(df_imr['Statut'], errors='coerce')
df_imr['Milieu_ecole'] = pd.to_numeric(df_imr['Milieu_ecole'], errors='coerce')

# 处理 目标数据 (df_outcome)
# 对应字段: Lib_DREN, Ecole, Statut, Milieu_ecole
df_outcome['key_dren'] = clean_key(df_outcome['Lib_DREN'])
df_outcome['key_ecole'] = clean_key(df_outcome['Ecole'])
df_outcome['Statut'] = pd.to_numeric(df_outcome['Statut'], errors='coerce')
df_outcome['Milieu_ecole'] = pd.to_numeric(df_outcome['Milieu_ecole'], errors='coerce')


# 在 merge 之前，必须保证 df_imr 中的关联键是唯一的，否则，如果 df_imr 有重复行，merge 会导致 df_outcome 行数爆炸。
merge_keys = ['key_dren', 'key_ecole', 'Statut', 'Milieu_ecole']

# 提取需要的列，并针对关联键去重 (保留第一个计算出的 IMR)
df_imr_unique = df_imr[merge_keys + ['IMR']].drop_duplicates(subset=merge_keys)

print(f"IMR表去重后行数: {len(df_imr_unique)} (用于匹配)")

# --- 4. 执行合并 (Merge) ---

# 使用左连接 (Left Join)，保留所有 outcomes 数据，匹配不上的 IMR 设为 NaN
df_final = pd.merge(
    df_outcome,
    df_imr_unique,
    on=merge_keys,
    how='left'
)

# 删除辅助列
df_final = df_final.drop(columns=['key_dren', 'key_ecole'])

# 检查匹配情况
matched_count = df_final['IMR'].notna().sum()
total_count = len(df_final)
print(f"\n匹配完成。")
print(f"总行数: {total_count}")
print(f"成功匹配 IMR 的行数: {matched_count} ({matched_count/total_count:.2%})")

########################################################################################################
#匹配方式：学校名、学校所在区域等元素组成的元组匹配
#这一段直接把参加了项目但是没匹配上的学校的IMR填成平均值。
# imr_mean = df_final['IMR'].mean()
# df_final['IMR'] = df_final['IMR'].fillna(imr_mean)


#drop参加了项目但是在总学校表里没匹配上的学校
df_final = df_final.dropna(subset=['IMR'])
print(f"除去参加了项目但是在总学校表里没匹配上的学校后的行数：{len(df_final)}")
########################################################################################################

# 保存结果
output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final_with_IMR.csv'
df_final.to_csv(output_path, index=False)
print(f"文件已保存至: {output_path}")

数据加载完成。IMR源数据行数: 19495, 结果数据行数: 43632
IMR表去重后行数: 19269 (用于匹配)

匹配完成。
总行数: 43632
成功匹配 IMR 的行数: 42019 (96.30%)
除去参加了项目但是在总学校表里没匹配上的学校后的行数：42019
文件已保存至: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final_with_IMR.csv


# 草稿

In [12]:
# 定义用于匹配的字段
# TARGET_COLS = ['DRENA', 'Nom_ecole', 'Statut', 'Milieu_ecole']
TARGET_COLS = ['DRENA','Nom_ecole', 'Statut', 'Milieu_ecole']
PROJECT_COLS = ['Lib_DREN', 'Ecole', 'Statut', 'Milieu_ecole']

df_project = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final.csv')
df_project_clean = df_project[PROJECT_COLS].apply(lambda x: x.dropna().astype(str).str.strip())
all_project_schools = set(df_project_clean.itertuples(index=False, name=None))


# 创建用于比对的四元组集合
df_Sumschool_clean = df_Sumschool[TARGET_COLS].apply(lambda x: x.dropna().astype(str).str.strip())
school_tuples = set(df_Sumschool_clean.itertuples(index=False, name=None))


print(len(df_project_clean))
print(len(all_project_schools))

print(len(df_Sumschool_clean))
print(len(school_tuples))





43632
3333
19495
19269


In [13]:
Sumschool_file = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\Ecoles_primaires_2024_2025.xlsx'
df_Sumschool =pd.read_excel(Sumschool_file, sheet_name=0)



all_project_schools = set(df_Sumschool['Nom_ecole'].dropna().astype(str).str.strip())
all_project_schools_havespace= set(df_Sumschool['Nom_ecole'])


print(sorted(list(all_project_schools)))
print(sorted(list(all_project_schools_havespace)))
print(len(all_project_schools))
print(len(all_project_schools_havespace))

['ACCADEMIE LA BONNE VOIE', 'ADJAME GUIPRY 2', 'ATTA ZIB WA TAALIM', 'BAGBA EXTENS 3', 'BERLIOZ DE BINGERVILLE', 'CCPPE LES ELITES', 'CEC  KONE TELE', 'CEC ARC EN CIEL', 'CEC CHRIST ROI DE FRANCOISKRO', 'CEC DEUTERONOME DE OLLOKRO', 'CEC DJAMANOUROU', 'CEC DRISSAKAHA', 'CEC ESPERANCE GARAHIO', 'CEC GROUPE SCOLAIRE  OTEME', 'CEC JEANNE DAGROU', 'CEC JESUFLA', 'CEC LA CITE DES ANGES', 'CEC LA PEPINIERE ST THOMAS', 'CEC LA REGIONALE AHIZABRE', 'CEC LES JULIENNES', 'CEC LES PETITS ANGES', 'CEC MOUSSAKAHA', 'CECILE MONGNETENOU', "CENTRE D'EDUCATION ISLAMIQUE", 'CENTRE MATERNELLE PAULINE KEREGOMARD', 'CPC DJAKOBOU', 'CPC DOSSONGUIKAHA 2', 'CPC LA PROVIDENCE', 'CPC LA REFERENCE DE PETIT MANKONO', 'CPC LADJIDOUGOU', 'CPC LAMADOUGOU', 'CPC LANDRIKRO', 'CPC MORIFEREDOUGOU', 'CPC SIRIKIKAHA', "DAR EL CORAN D'ANOUGBAKRO", "DAR ELFIKIH D'AMELEKIA", 'DAR FATOUA DE TAHAKRO', "DJON N'GUESSAN DAVID", 'E C AHOSSIKRO', 'E C FRANÇOIKRO', 'EBEN-EZER', 'EC 1 DE GBANGBOKOUADIOKRO', 'EC 2 DE GBANGBOKOUADIOKRO

In [14]:
file_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final.csv'
df = pd.read_csv(file_path)


all_project_schools = set(df['Ecole'].dropna().astype(str).str.strip())
all_project_schools_havespace= set(df['Ecole'])


print(sorted(list(all_project_schools)))
print(sorted(list(all_project_schools_havespace)))
print(len(all_project_schools))
print(len(all_project_schools_havespace))

['EPC  BAPTISTE AEEBCI KANOROBA', 'EPC 1 BEOUMI', 'EPC 2 BEOUMI', 'EPC 2 SAKASSOU', 'EPC ABOBO DOUME METHODISTE', 'EPC ADVENTISTE', 'EPC AGNI-FILLES', 'EPC AGNI-GARÇONS', 'EPC AL AMINE', 'EPC AL FADIL', 'EPC AL MOUSTAPHA DE LOFLA', 'EPC ANASS BOUN MALICK', 'EPC ASSAKAFATOUL ISLAMIATOU', 'EPC ASSEMBLEE DE DIEU', 'EPC ASSEMBLEE DE DIEU 1', 'EPC ASSEMBLEE DE DIEU ANTOICHE', 'EPC ASSEMBLEE DE DIEU DE GNIPI 2', 'EPC ATTABIATOUL ISLAMIYA 2 DE BAKARIDOUGOU', 'EPC ATTAHZIB WATTALIM', 'EPC ATTARBIAT ISLAMIA APPROMPRON', 'EPC ATTAZIB WATTAALIMI', 'EPC AZZAHRWANE', 'EPC BAPTISTE AEBECI BETHANIE', 'EPC BAPTISTE AEBECI DIANRA', 'EPC CATHOLIQUE 1', 'EPC CATHOLIQUE 2', 'EPC CATHOLIQUE NDA DUEKOUE', 'EPC CENTRE MOUHAMED BEN OUSSEYMINE', 'EPC CHEICK DIANE ALASSANE ISLAMIA', 'EPC CHEICK MOCTAR KAMATE', 'EPC CHEICK OUMAR', 'EPC CHERIFLA', 'EPC DA AWATOU ISLAMIYAT', 'EPC DAME', "EPC DAR AL QU'RAN AL KARIM/ANYAMA", 'EPC DAR ALIHIKMATOU WAL BAYANE', 'EPC DAR EL FIKHI ISLAMIA DE KILOKLE', 'EPC DAR EL HADIS',